In [1]:
# run_my_scenarios_with_prof_solver.py
# 目的：
# 1) 让你在这里 DIY 小规模场景（无需 CSV）；
# 2) 自动把场景转成导师代码所需的全局：num_scenarios / df / sp / plot_dir；
# 3) 复用导师的 build_pid_model + GurobiLBLowerBounder + snoglode 求解；
# 4) 用包装器把每个场景的概率设定为你在 scenarios 里给的 "prob"。

import os
import numpy as np
import pandas as pd
import pyomo.environ as pyo

# ========= 修改这里：填入你导师那份包含 build_pid_model(...) 的模块名 =========
# 例如：导师文件叫 stochastic_pid_prof.py，则 PROF_MODULE = "stochastic_pid_prof"
PROF_MODULE = "aaa"   # <<< 改成实际模块名

# --------------- 导入导师代码（build_pid_model / 下界类 / snoglode） ---------------
prof = __import__(PROF_MODULE, fromlist=["*"])
# 需要的对象：build_pid_model, GurobiLBLowerBounder, sno, get_solver/ipopt 工具等
build_pid_model_prof = prof.build_pid_model
GurobiLBLowerBounder = prof.GurobiLBLowerBounder
sno = prof.sno
get_solver = prof.get_solver
ipopt = get_solver("ipopt")

# ============= 你的可配置部分（DIY 场景、时间轴等） =============
# 统一的时间轴设置：总时长与离散段数；导师示例用 T=15, nfe=20（步长 0.75）
T_HORIZON = 15.0
NFE = 20
H = T_HORIZON / NFE
times = [i * H for i in range(NFE + 1)]  # 0, H, 2H, ..., T_HORIZON

# 自定义扰动与设定值（示例函数；你也可直接给 list）
def step_sp(t, step_time=3.0, low=0.0, high=0.5):
    return high if t >= step_time else low

def make_d(level=0.2, start=5.0, end=10.0):
    # 在 [start, end] 区间施加常量扰动 level
    return [level if (t >= start and t <= end) else 0.0 for t in times]

# -------------- 在这里 DIY 你的 scenarios（与示例一致） --------------
scenarios = {
    1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
    2: {"prob": 0.4, "Ku": 2.7, "tau": 1.8, "d": make_d(0.5), "sp": [step_sp(t) for t in times]},
    3: {"prob": 0.2, "Ku": 3.3, "tau": 2.2, "d": make_d(0.8), "sp": [step_sp(t) for t in times]},
}
# ===============================================================

# ============= 适配层：把 scenarios → 导师代码的全局 =============
# 导师 build_pid_model(scenario_name) 期望：
# - 全局 num_scenarios, df（含 tau_xs, tau_us, tau_ds, disturbance_0..NFE）, sp（常数）
# - scenario_name 形如 "scen_i"，它会用此名字去 df.iloc[...] 取行
#
# 注意：导师示例里 setpoint_change 最终被写死为全局 sp（常数）。
# 若你给了时变 sp[t]，此处暂取 sp_const = sp_seq[0] 用于所有场景（与导师示例对齐）。
# 如需时变设定值，需要改导师函数本体，将 x_setpoint 改为 Param(m.t)。

def _resample_to_grid(values, dst_len):
    """把任意长度序列线性重采样到目标长度 dst_len（一般为 NFE+1）。"""
    values = list(values)
    if len(values) == dst_len:
        return [float(v) for v in values]
    x_src = np.linspace(0.0, 1.0, len(values))
    x_dst = np.linspace(0.0, 1.0, dst_len)
    return list(np.interp(x_dst, x_src, values))

def prepare_prof_globals_from_scenarios(scenarios_dict, nfe=NFE, times_list=times):
    rows = []
    scen_names = []

    # 取全局常数 setpoint（与导师代码一致：他们用常数 sp）
    first_key = next(iter(scenarios_dict.keys()))
    first_sp = scenarios_dict[first_key]["sp"]
    sp_const = first_sp[0] if isinstance(first_sp, (list, tuple, np.ndarray)) else float(first_sp)

    # 0 基 scen_i 命名
    for idx, key in enumerate(sorted(scenarios_dict.keys())):
        s = scenarios_dict[key]
        Ku, tau = float(s["Ku"]), float(s["tau"])
        d_seq = _resample_to_grid(s["d"], nfe + 1)

        # 三系数映射：x' = -tau_xs x + tau_us u + tau_ds d
        tau_xs = 1.0 / tau
        tau_us = Ku / tau
        tau_ds = 1.0 / tau

        row = {
            "tau_xs": tau_xs,
            "tau_us": tau_us,
            "tau_ds": tau_ds,
            "scenario_name": f"scen_{idx}",
            "setpoint_change": float(sp_const),  # 虽然导师函数里没直接用这列，但留着对齐
            "probability": float(s.get("prob", 1.0 / len(scenarios_dict))),
        }
        for i in range(nfe + 1):
            row[f"disturbance_{i}"] = float(d_seq[i])

        rows.append(row)
        scen_names.append(row["scenario_name"])

    # 写入导师模块的全局变量（他们的 build_pid_model 会读取这些）
    prof.num_scenarios = len(rows)
    prof.df = pd.DataFrame(rows)
    prof.sp = float(sp_const)  # 关键：导师代码内部把 setpoint 当成全局常数 sp
    prof.plot_dir = os.getcwd() + "/plots_snoglode_parallel/"
    os.makedirs(prof.plot_dir, exist_ok=True)

    return scen_names


# ============= 概率包装器：把导师默认概率替换为 scenarios 里给的 prob =============
def build_pid_model_with_prob(scenario_name):
    m, first_stage, _prob_default = build_pid_model_prof(scenario_name)
    _, scen_num_str = scenario_name.split("_")
    scen_idx = int(scen_num_str)          # 0 基
    row = prof.df.iloc[scen_idx]          # <<< 不再减 1
    my_prob = float(row["probability"])
    return [m, first_stage, my_prob]


# ============= 主流程：准备 → 配置求解器 → 开始求解 =============
def main():
    # 1) 把你的 scenarios 转成导师代码用的全局
    scen_names = prepare_prof_globals_from_scenarios(scenarios, nfe=NFE, times_list=times)

    # 2) 创建 snoglode 参数（与导师脚本一致，只改 subproblem_creator）
    nonconvex_gurobi = pyo.SolverFactory("gurobi")
    nonconvex_gurobi.options["NonConvex"] = 2

    nonconvex_gurobi_lb = pyo.SolverFactory("gurobi")
    nonconvex_gurobi_lb.options["NonConvex"] = 2
    nonconvex_gurobi_lb.options["MIPGap"] = 0.2
    nonconvex_gurobi_lb.options["TimeLimit"] = 15

    obbt_solver_opts = {
        "NonConvex": 2,
        "MIPGap": 1,
        "TimeLimit": 5
    }

    # 用我们的包装器替代导师原来的 build_pid_model，使其返回自定义概率
    params = sno.SolverParameters(
        subproblem_names = scen_names,
        subproblem_creator = build_pid_model_with_prob,  # <<<< 关键：换成包装器
        lb_solver = nonconvex_gurobi_lb,
        cg_solver = ipopt,
        ub_solver = nonconvex_gurobi
    )
    params.set_bounders(candidate_solution_finder = sno.SolveExtensiveForm,
                        lower_bounder = GurobiLBLowerBounder)
    params.set_bounds_tightening(fbbt=True, obbt=True, obbt_solver_opt=obbt_solver_opts)
    params.set_branching(selection_strategy = sno.HybridBranching,
                         partition_strategy = sno.ExpectedValue)
    params.activate_verbose()

    # 3) 求解
    solver = sno.Solver(params)
    solver.solve(max_iter=1000, rel_tolerance=1e-3, time_limit=600*6)

    # 4)（可选）打印一阶段解并画图（直接复用导师脚本中那段）
    #    这里略；如需完全同款输出，可把导师文件末尾那段拷贝过来使用 solver.solution。

if __name__ == "__main__":
    main()


Generating the models for the subproblems.
	Scenario scen_0 has 3 first stage vars.
	Scenario scen_1 has 3 first stage vars.
	Scenario scen_2 has 3 first stage vars.
Finished generating subproblem models.
Node rooted.
Using EF method for candidate solution. No EF bound. Building EF.
EF built.
        Time (s)  Nodes Explored       Pruned by                              LB              UB        Rel. Gap        Abs. Gap         # Nodes
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
           1.175               1                           * L U     0.001225622     0.001442975        15.0628%        0.000217               2
SNoGloDe converged - absolute gap tolerance met.
